# สาธิต Inference: จำแนกตัวอักษรไทย 72 คลาส

โน้ตบุ๊กนี้ใช้ **DenseNet121 E1 candidate** และ weight ที่อยู่ใน repository เพื่อทำนายภาพใหม่ โดยเรียก `src.inference` ชุดเดียวกับโปรแกรมคำสั่ง จึงใช้การแปลงภาพและลำดับ label เดียวกัน **ไม่ต้องเทรนใหม่**

กด **Run All** แล้วจะเห็นภาพตัวอย่าง ข้อมูลโมเดล คำทำนาย Top-3 และได้ไฟล์ CSV ที่ส่งต่อได้ ภาพตัวอย่าง `ก` มาจากชุดข้อมูลพัฒนา ใช้ตรวจว่าระบบทำงานเท่านั้น **ไม่ใช่ผลประเมินกับข้อมูลอิสระ**

## 1. เตรียมเครื่องและเลือกภาพ

จาก root ของ repository ให้ติดตั้ง dependencies ครั้งแรก แล้วเลือก Python kernel จาก `.venv` ใน VS Code หรือ Jupyter:

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements-training.txt
```

ค่า `INPUT_PATH` ด้านล่างชี้ไปที่ภาพตัวอย่างที่มากับ repo จึงรันได้ทันทีหลัง clone หากต้องการใช้ภาพของตัวเอง เปลี่ยนเป็น path ของ **ไฟล์ภาพหนึ่งไฟล์** หรือ **โฟลเดอร์ภาพ**; path สัมพัทธ์เริ่มจาก root ของ repo รองรับ PNG, JPG, JPEG, BMP, TIFF และ WebP ภาพหนึ่งไฟล์ควรมีตัวอักษรเดี่ยวหนึ่งตัว

In [ ]:
INPUT_PATH = "notebooks/assets/example_ก.png"  # เปลี่ยนเป็นรูปหรือโฟลเดอร์ของคุณ
DEVICE = "cpu"  # เปลี่ยนเป็น auto เพื่อเลือก CUDA/MPS เมื่อมี
TOP_K = 3  # แสดงคำตอบที่เป็นไปได้มากที่สุด 3 อันดับ
CONFIDENCE_THRESHOLD = 0.0  # คะแนนต่ำกว่านี้จะติดสถานะ LOW_CONFIDENCE
OUTPUT_CSV = "results/predictions/inference_demo.csv"

## 2. โหลดโมเดลและตรวจข้อมูลกำกับ

ไฟล์ `best_model.pt`, `best_config.json` และ `label_to_index.json` อยู่ใน `results/inference/densenet121_e1_candidate/` ทั้งหมด โน้ตบุ๊กตรวจว่า config และ label map ตรงกับที่ฝังใน checkpoint ก่อนทำนาย และแสดง SHA-256 เพื่อระบุว่าใช้ weight ตัวใด

ภาพจะถูกแปลงเป็น RGB, ย่อโดยรักษาสัดส่วน, เติมพื้นขาวให้เป็น 128×128, แปลงเป็น tensor และ normalize แบบ ImageNet ไม่มี random augmentation ตอน inference

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
from PIL import Image

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/inference.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("เปิด notebook จากภายในโฟลเดอร์ repo")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from src.inference import Predictor, discover_defaults, discover_images, save_csv

weights, config, labels = discover_defaults()
predictor = Predictor(weights, device=DEVICE, config_path=config, labels_path=labels,
                      confidence_threshold=CONFIDENCE_THRESHOLD)
display(pd.DataFrame({"รายการ": ["สถาปัตยกรรม", "จำนวนคลาส", "ขนาดภาพเข้าโมเดล", "อุปกรณ์", "SHA-256 checkpoint"],
                      "ค่า": [predictor.config.architecture, len(predictor.labels),
                              f"{predictor.config.image_size} × {predictor.config.image_size}",
                              str(predictor.device), predictor.sha256]}))

## 3. ตรวจภาพที่จะทำนาย

สำหรับไฟล์เดียว โน้ตบุ๊กแสดงภาพแบบขยายเพื่อให้ดูง่าย แต่โมเดลยังรับ **ไฟล์ต้นฉบับ** สำหรับโฟลเดอร์ ระบบจะค้นหารูปในโฟลเดอร์ย่อยด้วยและเรียงลำดับ path ก่อนทำนาย

In [ ]:
if not INPUT_PATH:
    raise ValueError("ตั้ง INPUT_PATH เป็นไฟล์ภาพหรือโฟลเดอร์ภาพก่อนรัน")
input_path = Path(INPUT_PATH).expanduser()
if not input_path.is_absolute():
    input_path = repo_root / input_path
image_paths = discover_images(input_path)
if not image_paths:
    raise ValueError(f"ไม่พบไฟล์ภาพใน {input_path}")
print(f"อินพุต: {input_path} | จำนวนภาพ: {len(image_paths)}")
if input_path.is_file():
    with Image.open(input_path) as image:
        display(image.resize((192, 192), Image.Resampling.NEAREST))
else:
    display(pd.DataFrame({"ตัวอย่างไฟล์ (สูงสุด 10)": [p.relative_to(input_path).as_posix() for p in image_paths[:10]]}))

## 4. ทำนายและบันทึกผล

รูปเดี่ยวจะแสดงอันดับ, label และ confidence; ถ้าเป็นโฟลเดอร์จะแสดงตัวอย่างผล 20 แถว ทั้งสองกรณีบันทึก CSV ที่ `OUTPUT_CSV` โดยใช้รูปแบบเดียวกับ CLI ภาพที่อ่านไม่ได้จะเป็นแถว `ERROR` แทนที่จะทำให้ทั้งโฟลเดอร์หยุด

In [ ]:
predictions = predictor.predict(image_paths, top_k=TOP_K)
csv_path = Path(OUTPUT_CSV).expanduser()
if not csv_path.is_absolute():
    csv_path = repo_root / csv_path
save_csv(csv_path, predictions)
print(f"บันทึกผล {len(predictions)} ภาพ: {csv_path}")

if input_path.is_file():
    result = predictions[0]
    if result["status"] == "ERROR":
        raise ValueError(result["error"])
    display(pd.DataFrame([{"อันดับ": rank, "label": item["label"],
                           "confidence": f"{item['confidence']:.2%}"}
                          for rank, item in enumerate(result["top_k"], 1)]))
    print(f"ผลหลัก: {result['prediction']} | สถานะ: {result['status']} | เวลา: {result['inference_ms']:.1f} ms")
    if result["margin"] is not None:
        print(f"ส่วนต่างอันดับ 1–2: {result['margin']:.2%}")
else:
    display(pd.DataFrame([{"ไฟล์": row["filename"], "ผลทำนาย": row["prediction"],
                           "confidence": row["confidence"], "สถานะ": row["status"]}
                          for row in predictions[:20]]))
    print(f"สำเร็จ: {sum(row['status'] != 'ERROR' for row in predictions)} | อ่านไม่ได้: {sum(row['status'] == 'ERROR' for row in predictions)}")

## 5. อ่านผลให้ถูกต้อง

| ช่องข้อมูล | ความหมาย |
|---|---|
| `prediction` | label ที่ได้อันดับหนึ่ง เช่น `0_ก`: เลขหน้าเป็นรหัสคลาสในชุดข้อมูล ตัวอักษรท้ายคืออักขระไทย |
| `confidence` | ค่า softmax ของอันดับหนึ่ง อยู่ในช่วง 0–1; ไม่ใช่ความน่าจะเป็นที่พิสูจน์แล้วว่าทายถูก |
| `top2_label`, `top3_label` | ทางเลือกอันดับสองและสาม; ตั้ง `TOP_K = 3` เพื่อให้มีครบ |
| `margin` | ผลต่าง confidence อันดับหนึ่งกับสอง; ว่างเมื่อ `TOP_K = 1` |
| `status` | `OK`, `LOW_CONFIDENCE` ตาม threshold ที่ตั้ง หรือ `ERROR` ถ้าอ่านภาพไม่ได้ |
| `inference_ms` | เวลาเฉลี่ยต่อภาพใน batch นั้น ไม่รวมเวลาเปิด kernel และโหลดโมเดล |

ตัวอย่าง E1 ที่บันทึกไว้มี validation accuracy ประมาณ **99.80%** และ macro F1 ประมาณ **0.99796** บนชุด validation เดิม 86,103 ภาพ ([รายละเอียด](../docs/04-experiments.md)) ตัวเลขนี้ไม่ใช่คะแนนของภาพใหม่หรือชุดทดสอบภายนอก และการทำนายภาพตัวอย่างในโน้ตบุ๊กไม่ใช่การประเมินความแม่นยำ

## 6. ขอบเขตและวิธีส่งต่อ

- ใช้กับ **ภาพตัวอักษรเดี่ยว**; ภาพคำหรือประโยคยังไม่มีขั้นตรวจจับและตัดตัวอักษรที่พร้อมใช้งาน
- checkpoint นี้เป็น **E1 candidate** ที่พร้อมรัน inference แต่ยังไม่ใช่โมเดลสุดท้ายหลังการเปรียบเทียบ E1/E2
- เพื่อน clone repo, ติดตั้ง dependencies, เลือก kernel `.venv`, เปลี่ยน `INPUT_PATH` แล้วกด Run All ได้เลย ไม่ต้องคัดลอกชุดข้อมูลฝึก
- CSV ที่สร้างใน `results/predictions/` ถูก ignore โดย Git; ถ้าจะส่งผลให้คนอื่น ให้ส่ง CSV แยก หรือเลือก `OUTPUT_CSV` เป็น path ที่ต้องการ
- ใช้ผ่าน terminal ได้เช่นกัน: `python -m src.inference --input path/to/image.png --device cpu --top-k 3`